In [1]:
#laoding necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import tensorflow.keras.backend as K

In [2]:
# Loading preprocessed data
train = pd.read_csv("train_cleaned.csv")

# Split features and target
X = train.drop(columns=['Sales'])
y = train['Sales']

# Log-transform the target
y_log = np.log1p(y)

# Train/Validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# Scale features
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)



In [3]:
# RMSPE metric
def rmspe(y_true, y_pred):
    y_true_exp = tf.math.expm1(y_true)
    y_pred_exp = tf.math.expm1(y_pred)
    pct_error = (y_true_exp - y_pred_exp) / K.clip(y_true_exp, K.epsilon(), None)
    return K.sqrt(K.mean(K.square(pct_error)))

In [ ]:
#Modeling

# DNN Model
#model architecture
dnn = Sequential([
    Dense(200, activation='relu', input_dim=X_train_scaled.shape[1]),
    Dense(100, activation='relu'),
    Dense(50, activation='relu'),
    Dense(1, activation='linear')
])
dnn.compile(optimizer=Adam(0.001), loss='mse', metrics=[rmspe])

es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
mc = ModelCheckpoint("best_dnn.weights.h5", save_best_only=True, save_weights_only=True)

dnn.fit(X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=50,
        batch_size=256,
        callbacks=[es, mc],
        verbose=1)

dnn.load_weights("best_dnn.weights.h5")
print("DNN RMSPE:", dnn.evaluate(X_val_scaled, y_val, verbose=0)[1])


# LSTM Model
#setting the number of time steps for our model
time_steps = 1
num_features= X_train_scaled.shape[1]

X_train_lstm = X_train_scaled.reshape(-1, time_steps, num_features)
X_val_lstm = X_val_scaled.reshape(-1, time_steps, num_features)
#model architecture
lstm = Sequential([
    LSTM(128, activation='relu', input_shape=(time_steps, num_features)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])
lstm.compile(optimizer=Adam(0.001), loss='mse', metrics=[rmspe])

es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
mc = ModelCheckpoint("best_lstm.weights.h5", save_best_only=True, save_weights_only=True)
#fitting our model
lstm.fit(X_train_lstm, y_train,
         validation_data=(X_val_lstm, y_val),
         epochs=50,
         batch_size=256,
         callbacks=[es, mc],
         verbose=1)
#laoding model best weights
lstm.load_weights("best_lstm.weights.h5")
print("LSTM RMSPE:", lstm.evaluate(X_val_lstm, y_val, verbose=0)[1])


# GRU Model
#gru model architecture
gru = Sequential([
    GRU(64, activation='relu', input_shape=(time_steps, num_features)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dense(32, activation='relu'),
    Dense(1)
])
gru.compile(optimizer=Adam(0.001), loss='mse', metrics=[rmspe])

es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
mc = ModelCheckpoint("best_gru.weights.h5", save_best_only=True, save_weights_only=True)
#model fitting
gru.fit(X_train_lstm, y_train,
        validation_data=(X_val_lstm, y_val),
        epochs=50,
        batch_size=256,
        callbacks=[es, mc],
        verbose=1)
#loading model best weights
gru.load_weights("best_gru.weights.h5")
print("GRU RMSPE:", gru.evaluate(X_val_lstm, y_val, verbose=0)[1])


/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-12-06 23:28:07.125282: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-12-06 23:28:07.125312: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-12-06 23:28:07.125323: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2025-12-06 23:28:07.125343: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-12-06 23:28:07.125352: I tensorflow/core/common_runtime/pluggable_device/pluggab

Epoch 1/50


2025-12-06 23:28:08.073632: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


2639/2639 ━━━━━━━━━━━━━━━━━━━━ 29s 10ms/step - loss: 0.5003 - rmspe: 0.9176 - val_loss: 0.6065 - val_rmspe: 0.5043
Epoch 2/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 25s 9ms/step - loss: 1.3548 - rmspe: 22696.8633 - val_loss: 0.3044 - val_rmspe: 0.4339
Epoch 3/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 25s 9ms/step - loss: 2.4416 - rmspe: 14137.6611 - val_loss: 2.3672 - val_rmspe: 0.7428
Epoch 4/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 25s 10ms/step - loss: 3.9455 - rmspe: 148439555899392.0000 - val_loss: 0.5924 - val_rmspe: 0.5641
Epoch 5/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 25s 9ms/step - loss: 4.7826 - rmspe: 654866816.0000 - val_loss: 1.5479 - val_rmspe: 3.0198
Epoch 6/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 27s 10ms/step - loss: 5.8638 - rmspe: 21947020738560.0000 - val_loss: 0.2357 - val_rmspe: 0.5379
Epoch 7/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 25s 10ms/step - loss: 6.2458 - rmspe: 14140628992.0000 - val_loss: 1.5584 - val_rmspe: 0.6692
Epoch 8/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 26s 10ms/step - loss: 8.7053 - rmspe:

/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2639/2639 ━━━━━━━━━━━━━━━━━━━━ 41s 15ms/step - loss: 0.9916 - rmspe: 3.9599 - val_loss: 0.1276 - val_rmspe: 0.4172
Epoch 2/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 39s 15ms/step - loss: 0.1282 - rmspe: 0.4176 - val_loss: 0.1205 - val_rmspe: 0.3767
Epoch 3/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 38s 14ms/step - loss: 0.1167 - rmspe: 0.3956 - val_loss: 0.1074 - val_rmspe: 0.3663
Epoch 4/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 38s 14ms/step - loss: 0.1100 - rmspe: 0.3834 - val_loss: 0.1004 - val_rmspe: 0.3549
Epoch 5/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 40s 15ms/step - loss: 0.1067 - rmspe: 0.3760 - val_loss: 0.0974 - val_rmspe: 0.3414
Epoch 6/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 39s 15ms/step - loss: 0.1090 - rmspe: 0.3794 - val_loss: 0.1101 - val_rmspe: 0.3499
Epoch 7/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 41s 15ms/step - loss: 0.1739 - rmspe: 0.4831 - val_loss: 0.1444 - val_rmspe: 0.5175
Epoch 8/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 38s 14ms/step - loss: 0.3263 - rmspe: 0.7121 - val_loss: 0.2090 - val_rmspe: 0.4284
Epo

/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2639/2639 ━━━━━━━━━━━━━━━━━━━━ 55s 20ms/step - loss: 3.3183 - rmspe: 0.5793 - val_loss: 0.1434 - val_rmspe: 0.5076
Epoch 2/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 53s 20ms/step - loss: 0.1921 - rmspe: 0.5273 - val_loss: 0.1533 - val_rmspe: 0.3751
Epoch 3/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 50s 19ms/step - loss: 0.2082 - rmspe: 0.5561 - val_loss: 0.2396 - val_rmspe: 0.3816
Epoch 4/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 54s 20ms/step - loss: 0.2565 - rmspe: 0.6400 - val_loss: 0.2056 - val_rmspe: 0.6925
Epoch 5/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 54s 20ms/step - loss: 0.3252 - rmspe: 0.8066 - val_loss: 0.1326 - val_rmspe: 0.4111
Epoch 6/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 57s 22ms/step - loss: 0.3714 - rmspe: 0.9356 - val_loss: 0.1430 - val_rmspe: 0.4651
Epoch 7/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 55s 21ms/step - loss: 0.3687 - rmspe: 0.9181 - val_loss: 0.1604 - val_rmspe: 0.4962
Epoch 8/50
2639/2639 ━━━━━━━━━━━━━━━━━━━━ 59s 22ms/step - loss: 0.3925 - rmspe: 1.3432 - val_loss: 0.1719 - val_rmspe: 0.4810
Epo

The best performing model is Lstm with  0.330858051776886 rmspe.

hyperparameter tuning is commented out because it takes forever to run and to avoid problems rerunning the cells

In [ ]:
# import os

# os.makedirs("lstm_trials", exist_ok=True)

# #hyperparameter list 
# units_options = [64, 128, 256]
# dropout_options = [0.1, 0.2, 0.3]
# dense_options = [[64,32], [128,64], [128,64,32]]
# lr_options = [0.001, 0.0005]
# batch_options = [128, 256]

# # Prepare for results
# best_rmspe = float('inf')
# best_params = None
# best_model = None

# results = []
# trial_counter = 1





Hyperparameter tuning to get the best parameters.

In [ ]:

# # TIME_STEPS and input dimension
# time_steps = 1
# num_features = X_train_scaled.shape[1]

# #reshaping  for LSTM
# X_train_lstm = X_train_scaled.reshape(-1, time_steps, num_features)
# X_val_lstm = X_val_scaled.reshape(-1, time_steps, num_features)


In [ ]:

# for units, dropout, dense_units, lr, batch_size in product(
#     units_options, dropout_options, dense_options, lr_options, batch_options
#   ):
    
#     print(f"\nTrial {trial_counter}: units={units}, dropout={dropout}, dense={dense_units}, lr={lr}, batch={batch_size}")
    
#     # Building the model
#     lstm = Sequential()
#     lstm.add(LSTM(units, activation='relu', input_shape=(time_steps, num_features)))
#     lstm.add(Dropout(dropout))
#     for u in dense_units:
#         lstm.add(Dense(u, activation='relu'))
#     lstm.add(Dense(1))
    
#     lstm.compile(optimizer=Adam(lr), loss='mse', metrics=[rmspe])
    
#     # Callbacks
#     trial_path = f"lstm_trials/trial_{trial_counter}.weights.h5"
#     es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
#     mc = ModelCheckpoint(trial_path, save_best_only=True, save_weights_only=True)
    
#     # fitting model
#     lstm.fit(X_train_lstm, y_train,
#              validation_data=(X_val_lstm, y_val),
#              epochs=50,
#              batch_size=batch_size,
#              callbacks=[es, mc],
#              verbose=0)
    
#     # Load best weights
#     lstm.load_weights(trial_path)
    
#     # evaluating the model
#     val_rmspe = lstm.evaluate(X_val_lstm, y_val, verbose=0)[1]
#     print(f"RMSPE: {val_rmspe:.5f}")
    
#     # Saving the  results we get
#     results.append({
#         'trial': trial_counter,
#         'units': units,
#         'dropout': dropout,
#         'dense_units': dense_units,
#         'lr': lr,
#         'batch_size': batch_size,
#         'rmspe': val_rmspe,
#         'weights_path': trial_path
#     })
    
#     # Update the best results
#     if val_rmspe < best_rmspe:
#         best_rmspe = val_rmspe
#         best_params = (units, dropout, dense_units, lr, batch_size)
#         best_model = lstm
#         best_model.save("best_lstm_model.h5")  
    
#     trial_counter += 1

# #the best hyperparameters are units = 64, drouput = 0.1, dense_layers = [64,32],learning_rate = 0.001,batch_size = 128
# #we will comment it out to avoid it running everytime

Training our model with optimal hyperparameters from the gridsearch,units = 64, drouput = 0.1, dense_layers = [64,32],learning_rate = 0.001,batch_size = 128

In [ ]:
#we are going to build our model using the hyperparameters we got from tuning
#using gridsearch
time_steps = 1
num_features= X_train_scaled.shape[1]

# reshape data for LSTM
X_train_lstm = X_train_scaled.reshape(-1, time_steps, num_features)
X_val_lstm   = X_val_scaled.reshape(-1, time_steps, num_features)

# the model architecture
lstm = Sequential([
    LSTM(64, activation='relu', input_shape=(time_steps, num_features)),  
    Dropout(0.1),  
    Dense(64, activation='relu'),  
    Dense(32, activation='relu'),  
    Dense(1)
])

# compiling the model
lstm.compile(optimizer=Adam(0.001), loss='mse', metrics=[rmspe])

# callbacks
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
mc = ModelCheckpoint("best_lstm_tuned.weights.h5", save_best_only=True, save_weights_only=True)

# training our model
lstm.fit(
    X_train_lstm, y_train,
    validation_data=(X_val_lstm, y_val),
    epochs=50,
    batch_size=128,  # batch size from trial
    callbacks=[es, mc],
    verbose=1
)

#loading the best model weights saved during training
lstm.load_weights("best_lstm_tuned.weights.h5")
print("Tuned LSTM RMSPE:", lstm.evaluate(X_val_lstm, y_val, verbose=0)[1])


/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
5278/5278 ━━━━━━━━━━━━━━━━━━━━ 74s 14ms/step - loss: 0.6708 - rmspe: 0.9627 - val_loss: 0.1206 - val_rmspe: 0.4085
Epoch 2/50
5278/5278 ━━━━━━━━━━━━━━━━━━━━ 79s 15ms/step - loss: 0.1282 - rmspe: 0.4094 - val_loss: 0.2240 - val_rmspe: 0.6513
Epoch 3/50
5278/5278 ━━━━━━━━━━━━━━━━━━━━ 77s 15ms/step - loss: 0.1619 - rmspe: 0.4710 - val_loss: 0.1576 - val_rmspe: 0.5012
Epoch 4/50
5278/5278 ━━━━━━━━━━━━━━━━━━━━ 76s 14ms/step - loss: 0.2290 - rmspe: 0.5805 - val_loss: 1.1703 - val_rmspe: 3.5747
Epoch 5/50
5278/5278 ━━━━━━━━━━━━━━━━━━━━ 74s 14ms/step - loss: 0.3182 - rmspe: 0.7146 - val_loss: 0.5525 - val_rmspe: 1.5452
Epoch 6/50
5278/5278 ━━━━━━━━━━━━━━━━━━━━ 69s 13ms/step - loss: 0.4545 - rmspe: 1.1566 - val_loss: 0.2431 - val_rmspe: 0.7509
Tuned LSTM RMSPE: 0.39861536026000977


we see no improvement using the tuned hyperparameter we got from gridsearch.

In [ ]:
#We are going to try and tune a few hyperparameters here, like batch_size and optimizer
time_steps= 1
num_features = X_train_scaled.shape[1]

# reshaping data for LSTM model
X_train_lstm = X_train_scaled.reshape(-1, time_steps, num_features)
X_val_lstm   = X_val_scaled.reshape(-1, time_steps, num_features)

# model architecture
lstm = Sequential([
    LSTM(64, activation='relu', input_shape=(time_steps, num_features)),  
    Dropout(0.1),  
    Dense(64, activation='relu'),  
    Dense(32, activation='relu'),  
    Dense(1)
])

# compiling the model
lstm.compile(optimizer=Adam(0.005), loss='mse', metrics=[rmspe])

# callbacks
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
mc = ModelCheckpoint("tuned.weights.h5", save_best_only=True, save_weights_only=True)

# training our model
lstm.fit(
    X_train_lstm, y_train,
    validation_data=(X_val_lstm, y_val),
    epochs=50,
    batch_size=64 ,
    callbacks=[es, mc],
    verbose=1
)

# loading the best model weights saved during training
lstm.load_weights("tuned.weights.h5")
print(lstm.evaluate(X_val_lstm, y_val, verbose=0)[1])


/Users/ashadeen/Downloads/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
10555/10555 ━━━━━━━━━━━━━━━━━━━━ 152s 14ms/step - loss: 0.5586 - rmspe: 5892641.0000 - val_loss: 0.2171 - val_rmspe: 0.4861
Epoch 2/50
10555/10555 ━━━━━━━━━━━━━━━━━━━━ 152s 14ms/step - loss: 8.6236 - rmspe: inf - val_loss: 0.3318 - val_rmspe: 0.9513
Epoch 3/50
10555/10555 ━━━━━━━━━━━━━━━━━━━━ 149s 14ms/step - loss: 30.3533 - rmspe: inf - val_loss: 0.5380 - val_rmspe: 0.5436
Epoch 4/50
10555/10555 ━━━━━━━━━━━━━━━━━━━━ 152s 14ms/step - loss: 40.4670 - rmspe: inf - val_loss: 0.3674 - val_rmspe: 0.4309
Epoch 5/50
10555/10555 ━━━━━━━━━━━━━━━━━━━━ 155s 15ms/step - loss: 42.3731 - rmspe: inf - val_loss: 28.6194 - val_rmspe: 63810.6719
Epoch 6/50
10555/10555 ━━━━━━━━━━━━━━━━━━━━ 153s 15ms/step - loss: 50.7525 - rmspe: inf - val_loss: 40.9735 - val_rmspe: 0.9835
0.48113617300987244


In [ ]:
# We are going to try and tune a few hyperparameters here, like batch_size and optimizer
time_steps = 7   # number of past days to look at

# function to create sequences per store
def create_sequences(X, y, store_col, time_steps):
    X_seq, y_seq = [], []
    stores = np.unique(store_col)

    for store in stores:
        idx = (store_col == store)
        X_s = X[idx]
        y_s = y[idx]

        for i in range(time_steps, len(X_s)):
            X_seq.append(X_s[i-time_steps:i])
            y_seq.append(y_s[i])

    return np.array(X_seq), np.array(y_seq)


# converting data for LSTM input
train_seq, y_train_seq = create_sequences(
    X_train_scaled, 
    y_train.values, 
    X_train['Store'].values, 
    time_steps
)

val_seq, y_val_seq = create_sequences(
    X_val_scaled, 
    y_val.values, 
    X_val['Store'].values, 
   time_steps
)

num_features = train_seq.shape[2]


# model architecture
lstm = Sequential([
    LSTM(64, activation='relu', input_shape=(time_steps, num_features)),  
    Dropout(0.1),  
    Dense(64, activation='relu'),  
    Dense(32, activation='relu'),  
    Dense(1)
])

# compiling the model
lstm.compile(optimizer=Adam(0.005), loss='mse', metrics=[rmspe])

# callbacks
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
mc = ModelCheckpoint("tuned.weights.h5", save_best_only=True, save_weights_only=True)

# training our model
lstm.fit(
    train_seq, y_train_seq,
    validation_data=(val_seq, y_val_seq),
    epochs=50,
    batch_size=64,
    callbacks=[es, mc],
    verbose=1
)

# loading the best model weights saved during training
lstm.load_weights("tuned.weights.h5")
print(lstm.evaluate(val_seq, y_val_seq, verbose=0)[1])


2025-12-09 19:50:24.255521: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-12-09 19:50:24.255677: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-12-09 19:50:24.255692: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2025-12-09 19:50:24.255881: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-12-09 19:50:24.255891: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
/Users/ashadeen/Downloads/2501218_ASHA_DEEN_CE889/rossmann-store-sales/rossmann_env/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape

Epoch 1/50


2025-12-09 19:50:25.283734: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


10433/10433 ━━━━━━━━━━━━━━━━━━━━ 440s 42ms/step - loss: 566762.5625 - rmspe: inf - val_loss: 12698.7227 - val_rmspe: inf
Epoch 2/50
10433/10433 ━━━━━━━━━━━━━━━━━━━━ 831s 80ms/step - loss: 44688.1641 - rmspe: inf - val_loss: 0.1854 - val_rmspe: 0.5053
Epoch 3/50
10433/10433 ━━━━━━━━━━━━━━━━━━━━ 24220s 2s/step - loss: 519548.1562 - rmspe: inf - val_loss: 264933.1875 - val_rmspe: inf
Epoch 4/50
10433/10433 ━━━━━━━━━━━━━━━━━━━━ 412s 40ms/step - loss: 944491328.0000 - rmspe: inf - val_loss: 2934862592.0000 - val_rmspe: inf
Epoch 5/50
10433/10433 ━━━━━━━━━━━━━━━━━━━━ 413s 40ms/step - loss: 1019585728.0000 - rmspe: inf - val_loss: 325693.7188 - val_rmspe: 1.0002
Epoch 6/50
10433/10433 ━━━━━━━━━━━━━━━━━━━━ 414s 40ms/step - loss: 1106777216.0000 - rmspe: inf - val_loss: 24317794.0000 - val_rmspe: 1.0002
Epoch 7/50
10206/10433 ━━━━━━━━━━━━━━━━━━━━ 8s 38ms/step - loss: 1019369239.5783 - rmspe: inf